In [5]:
import os
import pandas as pd
import pyodbc
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.dataset as ds
import pyarrow.compute as pc

warnings.filterwarnings('ignore')

usuario = os.getenv('USERNAME')
path_senhas = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'

df_senhas = pd.read_excel(fr'{path_senhas}\SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

def faixas_score_pf():
    intervalos = [0, 337, 416, 476, 524, 567, 607, 651, 701, 1000]
    lista_rating = ['H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', 'AA']
    return intervalos, lista_rating

def define_rating(score, intervalos, lista_rating):
    for i in range(len(intervalos) - 1):
        if intervalos[i] < score <= intervalos[i + 1]:
            return lista_rating[i]
    return None

def tratando_documento(val):
    val = float(val)
    val = str(val).split('.')[0]
    return val

def safra_alocacao(val):
    val = str(val)[0:7]
    return val

# 1. Consulta base 2
query_base2 = """
SELECT
    a.ID_Cota,
    a.DT_Alocacao,
    a.Tipo_Pessoa,
    c.NM_Produto as Nome_Produto,
    b.CD_InscricaoNacional AS CPF_CNPJ
FROM [dbo].[FT0002_VendaAlocacoes] a
LEFT JOIN [dbo].[DM0013_Pessoas] b ON a.ID_Pessoa = b.ID_Pessoa
LEFT JOIN [dbo].[DM0002_Produtos] c on a.ID_Produto = c.ID_Produto
where a.DT_Alocacao >= '2024-01-01'
and a.DT_Alocacao <= '2025-04-30'
and a.Tipo_Pessoa = 'F'
"""

print("📥 Executando consulta Base 2 - alocação e pessoas...")
df_base2 = pd.read_sql(query_base2, conn)
df_base2['CPF_CNPJ'] = df_base2['CPF_CNPJ'].apply(tratando_documento)
df_base2['DT_Alocacao'] = pd.to_datetime(df_base2['DT_Alocacao'], format='%Y-%m-%d')

# 2. Carregar score
usuario_path = fr'C:\Users\{usuario}\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\MODELOS\MODELAGEM\SCORE_CREDITO_BOA_VISTA\BASES_HISTORICAS'
df_score = pd.read_csv(fr'{usuario_path}\score_credito_full.csv', delimiter=";")
df_score = df_score[df_score['Score_Credito'] > 2]
df_score['CPF_CNPJ'] = df_score['CPF_CNPJ'].apply(tratando_documento)
df_score['Data'] = pd.to_datetime(df_score['Data'], format='%Y-%m-%d')

df_base2 = df_base2.merge(df_score, how='left', on='CPF_CNPJ')
df_base2 = df_base2[df_base2['Score_Credito'].notna()]
df_base2['Diferenca_Datas'] = abs(df_base2['DT_Alocacao'] - df_base2['Data']).dt.days
df_base2 = df_base2.sort_values(by=['Diferenca_Datas'])
df_base2 = df_base2.drop_duplicates(subset=['ID_Cota'], keep='first')
df_base2 = df_base2[df_base2['Diferenca_Datas'] <= 30]

# 3. Preparar para ler parquet com filtro de CNPJs PJ
path_parquet = r'C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\INDICADORES\NOVA ESTRUTURA\INADIMPLÊNCIA\INADIMPLÊNCIA GERAL\Dados\Dados - Ramo de Atividade\Principal\arquivo_unificado.parquet'

colunas_necessarias = ['CNPJ_BASICO', 'CNPJ_ORDEM', 'CNPJ_DV', 'SITUACAO_CADASTRAL', 'CNAE_FISCAL_PRINCIPAL']

# Lista de CNPJs únicos PJ para filtro
cnpjs_pj = df_base2[df_base2['Tipo_Pessoa'] == 'J']['CPF_CNPJ'].astype(str).str.zfill(14).unique().tolist()

def split_cnpj(cnpj_str):
    return cnpj_str[:8], cnpj_str[8:12], cnpj_str[12:14]

cnpj_tuplas = [split_cnpj(c) for c in cnpjs_pj]

dataset = ds.dataset(path_parquet, format="parquet")

# Construir filtro OR para os CNPJs
filtro = None
for basico, ordem, dv in cnpj_tuplas:
    cond = (
        (pc.equal(dataset.field('CNPJ_BASICO'), basico)) &
        (pc.equal(dataset.field('CNPJ_ORDEM'), ordem)) &
        (pc.equal(dataset.field('CNPJ_DV'), dv))
    )
    filtro = cond if filtro is None else (filtro | cond)

# Ler tabela filtrando colunas e aplicando filtro
table = dataset.to_table(
    columns=colunas_necessarias,
    filter=filtro
)

df_parquet = table.to_pandas()

# Processar CNPJ completo
df_parquet['CNPJ_BASICO'] = df_parquet['CNPJ_BASICO'].astype(str).str.zfill(8)
df_parquet['CNPJ_ORDEM'] = df_parquet['CNPJ_ORDEM'].astype(str).str.zfill(4)
df_parquet['CNPJ_DV'] = df_parquet['CNPJ_DV'].astype(str).str.zfill(2)
df_parquet['CNPJ'] = df_parquet['CNPJ_BASICO'] + df_parquet['CNPJ_ORDEM'] + df_parquet['CNPJ_DV']

ramo_map = {
    '01': 'AGRICULTURA, PECUÁRIA, PRODUÇÃO FLORESTAL, PESCA E AQÜICULTURA',
    '02': 'AGRICULTURA, PECUÁRIA, PRODUÇÃO FLORESTAL, PESCA E AQÜICULTURA',
    '03': 'AGRICULTURA, PECUÁRIA, PRODUÇÃO FLORESTAL, PESCA E AQÜICULTURA',
    '05': 'INDÚSTRIAS EXTRATIVAS',
    '06': 'INDÚSTRIAS EXTRATIVAS',
    '07': 'INDÚSTRIAS EXTRATIVAS',
    '08': 'INDÚSTRIAS EXTRATIVAS',
    '09': 'INDÚSTRIAS EXTRATIVAS',
    '10': 'INDÚSTRIAS DE TRANSFORMAÇÃO',
    '11': 'INDÚSTRIAS DE TRANSFORMAÇÃO',
    '12': 'INDÚSTRIAS DE TRANSFORMAÇÃO',
    '13': 'INDÚSTRIAS DE TRANSFORMAÇÃO',
    '14': 'INDÚSTRIAS DE TRANSFORMAÇÃO',
    '15': 'INDÚSTRIAS DE TRANSFORMAÇÃO'
}

df_parquet['Ramo_Atividade'] = df_parquet['CNAE_FISCAL_PRINCIPAL'].astype(str).str[:2].map(ramo_map)
df_parquet = df_parquet[['CNPJ', 'SITUACAO_CADASTRAL', 'Ramo_Atividade']]

# Merge só para PJ
df_pj = df_base2[df_base2['Tipo_Pessoa'] == 'J'].copy()
df_pj['CNPJ'] = df_pj['CPF_CNPJ'].astype(str).str.zfill(14)
df_pj = df_pj.merge(df_parquet, how='left', on='CNPJ')

df_pf = df_base2[df_base2['Tipo_Pessoa'] == 'F'].copy()
df_pf['SITUACAO_CADASTRAL'] = np.nan
df_pf['Ramo_Atividade'] = np.nan

df_base2 = pd.concat([df_pf, df_pj], ignore_index=True)

intervalos_pf, lista_rating_pf = faixas_score_pf()

df_silver_geral = df_base2[['DT_Alocacao', 'Tipo_Pessoa', 'CPF_CNPJ', 'Score_Credito', 'Ramo_Atividade']].copy()
df_silver_geral['Safra_Alocacao'] = df_silver_geral['DT_Alocacao'].apply(safra_alocacao)
df_silver_geral = df_silver_geral.drop(columns=['DT_Alocacao'])

df_silver_geral = df_silver_geral.groupby(['CPF_CNPJ', 'Tipo_Pessoa', 'Safra_Alocacao', 'Ramo_Atividade']).mean().reset_index()

def atribui_rating(row):
    if pd.isna(row['Score_Credito']):
        return None
    if row['Tipo_Pessoa'] == 'F':
        return define_rating(row['Score_Credito'], intervalos_pf, lista_rating_pf)
    return None

df_silver_geral['Rating'] = df_silver_geral.apply(atribui_rating, axis=1)

df_gold_geral = df_silver_geral[['Safra_Alocacao', 'Tipo_Pessoa', 'Rating', 'Ramo_Atividade']].copy()

ordem_rating = ['AA', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
cores = {
    'AA': '#0c2c84',
    'A':  '#225ea8',
    'B':  '#1d91c0',
    'C':  '#41b6c4',
    'D':  '#7fcdbb',
    'E':  '#c7e9b4',
    'F':  '#ffffcc',
    'G':  '#fdae61',
    'H':  '#d73027'
}

ramo_desejado = 'AGRICULTURA, PECUÁRIA, PRODUÇÃO FLORESTAL, PESCA E AQÜICULTURA'

df_filtrado = df_gold_geral[
    (df_gold_geral['Tipo_Pessoa'] == 'F') &
    (df_gold_geral['Ramo_Atividade'] == ramo_desejado)
].copy()

df_agrupado = (
    df_filtrado.groupby(['Safra_Alocacao', 'Rating'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=ordem_rating, fill_value=0)
    .sort_index()
)

totais_safra = df_agrupado.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
bottom = [0] * len(df_agrupado)
x_labels = df_agrupado.index.astype(str)

for rating in ordem_rating:
    valores = df_agrupado[rating]
    bars = ax.bar(x_labels, valores, bottom=bottom, color=cores[rating], label=rating)
    for i, (bar, valor) in enumerate(zip(bars, valores)):
        if valor > 0:
            pct = 100 * valor / totais_safra.iloc[i]
            ax.text(bar.get_x() + bar.get_width()/2,
                    bottom[i] + valor/2,
                    f"{pct:.1f}%",
                    ha='center', va='center',
                    fontsize=9, color='black', fontweight='bold')
    bottom = [b+v for b, v in zip(bottom, valores)]

ax.set_title(f"Concentração de Clientes PF vs Rating - Ramo: {ramo_desejado}")
ax.set_xlabel("Safra")
ax.set_ylabel("Quantidade de Clientes")
ax.legend(title="Rating", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


📥 Executando consulta Base 2 - alocação e pessoas...


MemoryError: Unable to allocate 1.00 MiB for an array with shape (1, 131072) and data type object